# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells
This notebook provides a template for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name: ", metadata.name)
print("Description: ", metadata.description)
print("Version: ", metadata.version)
print("Identifier: ", getattr(metadata, 'identifier', 'N/A'))
print("Published: ", getattr(metadata, 'datePublished', 'N/A'))


## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's list all record sets, their `@id`, and their fields with `@id` and descriptions.

In [ ]:
# Explore record sets and their fields, referencing all entities by their `@id`

# Ensure we use the metadata directly
record_sets = metadata.record_sets if hasattr(metadata, 'record_sets') else getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No record sets found in this dataset.")
else:
    for rs in record_sets:
        print(f"RecordSet: {rs['@id']}")
        print(f"  Name: {rs.get('name', 'N/A')}")
        # Print available fields
        if 'field' in rs:
            for field in rs['field']:
                print(f"    Field @id: {field['@id']}")
                print(f"      Name: {field.get('name','N/A')}")
                print(f"      Description: {field.get('description','N/A')}")
        else:
            print("    No fields found in this RecordSet.")
        print()

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the `@id` of record sets and fields discovered above.

_As an example, we'll attempt to extract all available record sets._

In [ ]:
# List all available record set IDs
# Run this if above visualization didn't show them explicitly
record_sets = metadata.record_sets if hasattr(metadata, 'record_sets') else getattr(metadata, 'recordSet', [])
record_set_ids = [rs['@id'] for rs in record_sets]
print("Available record_set @id's:", record_set_ids)

# Attempt to read the first record set (if present) as example
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records from record_set @id: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
    except Exception as e:
        print(f"  Could not load records from {rs_id}: {e}")
        continue
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  Loaded {len(df)} rows. Columns available: {df.columns.tolist()}")
    else:
        print(f"  No records found for {rs_id}.")

# Display first few rows from the first loaded DataFrame (if any)
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nShowing first 5 rows for record_set @id: {first_rs_id}")
    display(dataframes[first_rs_id].head())
else:
    print("No record sets could be loaded into DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates removing outliers, transforming distributions, or grouping data.

**Note:** Please update the code below to choose fields (`@id`) corresponding to a numeric field (e.g. peptide counts, coverage_percent) and a grouping field. You can change `numeric_field_id` and `group_field_id` values based on what's present in your dataset record set.

In [ ]:
import numpy as np

# Substitute these with actual field @id's from the print-outs above (example names shown)
record_set_id = first_rs_id if dataframes else None
# Example field @id, adjust as needed for your dataset
numeric_field_id = None
group_field_id = None

# If columns are present, choose numeric and group field for demonstration
if record_set_id and not numeric_field_id:
    # Try to guess a numeric field
    for col in dataframes[record_set_id].columns:
        # Common numeric field names in proteomics might include 'coverage', 'MW', 'peptide_count', etc.
        if any(kw in col.lower() for kw in ['coverage', 'mw', 'count', 'intensity', 'abundance', 'quant', 'score']):
            numeric_field_id = col
            break
    # Select a grouping field, try 'description', 'gene', or 'accession'
    for col in dataframes[record_set_id].columns:
        if col != numeric_field_id:
            if any(kw in col.lower() for kw in ['gene', 'protein', 'description', 'accession', 'type']):
                group_field_id = col
                break
if record_set_id and numeric_field_id:
    print(f"Using numeric field '@id': {numeric_field_id}")
    threshold = np.percentile(dataframes[record_set_id][numeric_field_id].dropna(), 75) # e.g., upper quartile
    filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (upper quartile): {len(filtered_df)} records")
    # Normalize
    filtered_df[numeric_field_id + "_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    display(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())
    # Grouping
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df[[numeric_field_id]].head())
else:
    print("Please set 'numeric_field_id' and 'group_field_id' based on available columns for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Update the field selections below according to your dataset fields (see the previous code block for selection logic).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id and numeric_field_id in dataframes[record_set_id].columns:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[record_set_id][numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
    # Boxplot by group (if grouping field is set)
    if group_field_id and group_field_id in dataframes[record_set_id].columns:
        # We'll cap to 10 groups to prevent overplotting
        top_groups = dataframes[record_set_id][group_field_id].value_counts().index[:10]
        plt.figure(figsize=(10, 6))
        sns.boxplot(
            data=dataframes[record_set_id][dataframes[record_set_id][group_field_id].isin(top_groups)],
            x=group_field_id, y=numeric_field_id
        )
        plt.title(f"{numeric_field_id} by {group_field_id} (Top 10 Groups)")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Please set 'numeric_field_id' for visualization steps.")

## 6. Conclusion

In this notebook, we loaded and explored the FAIR² dataset schema using the `mlcroissant` library, identifying available record sets and fields by their `@id`. We demonstrated data extraction, basic filtering, normalization, and grouped summarization. Visualizations illustrated the distributions of key fields. For further in-depth analysis or model-building, refer to the data dictionary (`@id` for fields/columns of interest) and use the `mlcroissant` API for robust, FAIR-aligned data access.

**Tips:**
- Always reference entities (record sets, fields) by their full `@id` for reproducibility.
- Explore additional dataset metadata: `metadata` contains rich descriptions and provenance references.
- For more complex querying, use Python logic to combine multiple field filters or join across datasets.
